In [1]:
import pandas as pd
from sqlalchemy import create_engine
import urllib
import re

In [2]:
params_data = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=181.57.189.150,1433;"
    "DATABASE=Ventas_Shopify;"
    "UID=sa;"
    "PWD=P@ssw0rd*;"
)

# Crear el motor de conexión
engine_data = create_engine(f"mssql+pyodbc:///?odbc_connect={params_data}")

# Configurar cadena de conexión
params_com = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=comerssiamirror.eastus2.cloudapp.azure.com,38693;"
    "DATABASE=PROVENZAL;"
    "UID=provenzal;"
    "PWD=PE@4:BRy/<1VWkp;"
)

# Crear el motor
engine_com = create_engine(f"mssql+pyodbc:///?odbc_connect={params_com}")

In [3]:
clientes_data = """
SELECT DISTINCT v.id_cliente AS Cliente
	FROM Ventas_ShopifyMovinova v
	WHERE  NOT EXISTS (SELECT 1 FROM Clientes_MARKETPLACE m WHERE m.id_Cliente = v.id_Cliente)
"""

# Ejecutar y cargar en DataFrame
clientes_shopify = pd.read_sql(clientes_data, engine_data)

clientes_com = """
SELECT DISTINCT 
   v.Cliente
FROM Clientes c
INNER JOIN Ventas_MoviNova v ON c.CLICodigo = v.Cliente
"""

# Ejecutar y cargar en DataFrame
df_clientes_comerssia = pd.read_sql(clientes_com, engine_com)


query_rappi = """
SELECT Cliente
FROM ClientesRAPPI 
"""

# Ejecutar y cargar en DataFrame
clientes_rappi = pd.read_sql(query_rappi, engine_com)

query_empleados = """
SELECT Codigo AS Cliente
FROM EmpleadosProvenzal
"""

# Ejecutar y cargar en DataFrame
empleados= pd.read_sql(query_empleados, engine_com)

query_marketplace = """
SELECT Id_Cliente AS Cliente
FROM Clientes_MARKETPLACE
"""

# Ejecutar y cargar en DataFrame
marketplace= pd.read_sql(query_marketplace, engine_data)

In [4]:
#Unir los Clientes a excluir
df_excluyentes = pd.concat([empleados, clientes_rappi], ignore_index=True)

#Eliminar clientes excluyentes.
df_clientes_comerssia = df_clientes_comerssia[~df_clientes_comerssia['Cliente'].isin(df_excluyentes['Cliente'])]
clientes_shopify = clientes_shopify[~clientes_shopify['Cliente'].isin(marketplace['Cliente'])]

#Unir clientes totales
df_clientes = pd.concat([df_clientes_comerssia, clientes_shopify], ignore_index=True)

def limpiar_codigo(codigo):
    codigo = str(codigo).strip()              # quita espacios al inicio y al final
    codigo = re.sub(r'^C\s*', '', codigo, flags=re.IGNORECASE)  # elimina 'C' + cualquier número de espacios al inicio
    codigo = codigo.replace(' ', '')          # elimina cualquier espacio restante en el medio
    return codigo

#Conviertir a String y aplicar funcion para lipiar codigo
df_clientes['Cliente'] = df_clientes['Cliente'].apply(limpiar_codigo)

df_clientes = df_clientes.drop_duplicates(subset=['Cliente'], keep='first')

In [5]:
df_clientes

,Cliente
0,28296992
1,1019115539
2,102546113
3,109872559
4,14940190
...,...
295311,886456
295318,1020801969
295320,1020758169
295321,4316552
